# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

Why this lane: the starter data is one row per page with 90-day search and analytics totals plus a 30-day vs previous-30-day comparison, so it can show which pages are visible *and* moving. That is the raw material for a review queue. The question is a prioritisation problem ("which page first?"), which fits a ranked list with reason codes better than a single yes/no prediction. Search impressions are also very concentrated in a small share of pages (numbers in section 3), so ordering the work matters.

I can confirm or change this lane until the end of Week 4. If the future-window labels in the warehouse turn out to be too noisy, Lane 4 (CTR / engagement opportunity) is my fallback because it uses the same data and a position-adjusted score.

**Search question:** *Among pages that already get meaningful search visibility, which should a reviewer look at first because their search impressions are falling?*

**One-paragraph frame:** For a content editor with limited review time, deciding which visible pages to look at first, I will build a ranked review queue with reason codes from observable search and analytics signals, scoring the risk that a page's impressions keep falling, measured by precision@K (compared with the base rate) on held-out clients. A wrong call costs reviewer time or a missed problem on a high-traffic page. A plain rule is a useful baseline but may not capture how many signals interact. I will claim only observed, directional and decision-support results.

In [1]:
import pandas as pd, numpy as np, os, urllib.request

PATHS = ["../../data/raw/content_refresh_anonymized.csv",   # local clone
         "data/raw/content_refresh_anonymized.csv"]          # repo root (Colab after clone)
path = next((p for p in PATHS if os.path.exists(p)), None)
if path is None:
    path = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)
print(f"{len(df):,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} pseudonymised clients")
print("one row = one content item (page); metrics are trailing 90 days")

30,000 rows x 44 columns, 32 pseudonymised clients
one row = one content item (page); metrics are trailing 90 days


## 2. The question: decision, action, cost of a wrong call

- **Unit of analysis:** one content item (page). In the starter CSV, one row per page; in the warehouse, one page at one decision date.
- **Decision improved:** which pages an editor reviews first, given limited capacity.
- **Who acts and what they do:** a content editor reads the top-K pages of the queue and chooses an action per page: refresh, review CTR or title/snippet, review on-page engagement, or monitor.
- **Output:** a ranked queue, with a score, a suggested review action, and reason codes (e.g. "visible page, impressions down, stale").
- **Cost of a wrong recommendation:**
  - *False positive* (a page is flagged but is fine, or a refresh would not help): wasted editor hours, and a better candidate waits.
  - *False negative* (a high-visibility page that is losing visibility is not flagged): the loss goes unnoticed for longer. This matters most for the few pages that carry much of the traffic.
  - Because errors on big pages cost more than on tiny pages, I will apply a minimum-volume rule and report precision at the capacity a reviewer really has (K = 20 or 50).
- **Why data/ML can help at all:** a hand-written rule (such as "stale and visible") is easy to explain and will be my baseline. Data may help where several signals (position, age, volume, CTR, engagement) interact in ways that are hard to write by hand. That is only worth keeping if it beats the rule on held-out clients. If it does not, the plain rule is the honest answer.
- **Not just "train a model":** the decision, the action, the capacity limit and the cost of errors come first. The model is one option for ordering the list, and the baseline rule is the thing it must beat.

In [2]:
# Review capacity vs how many candidates exist (a threshold is a policy choice, not a truth)
MIN_IMP = 500   # provisional "visible page" volume floor
visible = df[df["impressions_90d"] >= MIN_IMP]
visible_down = visible[visible["trend_direction"] == "down"]
print(f"Pages with >= {MIN_IMP} impressions/90d: {len(visible):,} ({len(visible)/len(df):.1%} of all pages)")
print(f"...of which labelled 'down' in the last 30 vs previous 30 days: {len(visible_down):,}")
print(f"A reviewer who can check 50 pages sees {50/len(visible_down):.2%} of that pool -> ordering matters")

Pages with >= 500 impressions/90d: 16,726 (55.8% of all pages)
...of which labelled 'down' in the last 30 vs previous 30 days: 9,961
A reviewer who can check 50 pages sees 0.50% of that pool -> ordering matters


## 3. Quick look at the data (2-3 real numbers)

The cell below computes the numbers from the starter CSV. Reading them:

1. **Search visibility is concentrated.** A small share of pages holds most of the impressions, so ordering review effort is worth doing: reviewing the right few pages covers a large share of exposure.
2. **Many visible pages show falling impressions on this label.** More than half of all pages, and a similar share of visible pages, are labelled `down`. A queue that flags "anything declining" would not narrow the list, so the ranking has to separate pages *within* that group. Note: `trend_direction` compares only two 30-day windows. I treat it as a rough proxy, not as a true "decline", and the capstone will move to a prior-window to future-window label.
3. **Clients differ a lot.** The share of declining pages varies widely between clients, so I must validate on held-out clients rather than random rows.

In [3]:
total_imp = df["impressions_90d"].sum()
s = df["impressions_90d"].sort_values(ascending=False)
for share in (0.05, 0.10):
    k = int(len(df) * share)
    print(f"[1] Top {share:.0%} of pages ({k:,}) hold {s.head(k).sum()/total_imp:.1%} of all impressions")

print(f"[2] Share labelled 'down': all pages {(df['trend_direction']=='down').mean():.1%}; "
      f"pages with >= {MIN_IMP} impressions {(visible['trend_direction']=='down').mean():.1%} "
      f"(base rate any ranking must beat)")

per_client = df.groupby("client_id")["trend_direction"].apply(lambda x: (x == "down").mean())
print(f"[3] Per-client 'down' share across {len(per_client)} clients: "
      f"min {per_client.min():.0%}, median {per_client.median():.0%}, max {per_client.max():.0%}")

# Reminder of what is NOT a feature (label source) - see data dictionary
print("Never features: trend_direction, trend_pct (they define the label). IDs are for grouping only.")

[1] Top 5% of pages (1,500) hold 54.1% of all impressions
[1] Top 10% of pages (3,000) hold 70.2% of all impressions
[2] Share labelled 'down': all pages 54.2%; pages with >= 500 impressions 59.6% (base rate any ranking must beat)
[3] Per-client 'down' share across 32 clients: min 0%, median 52%, max 94%
Never features: trend_direction, trend_pct (they define the label). IDs are for grouping only.


## 4. Careful words: what I can and can't claim

**What this work can say (observed / directional / decision-support):**
- *Observed:* in this 30,000-row starter slice, these are the measured patterns (concentration of impressions, share of pages labelled `down`, spread across clients).
- *Directional:* a signal that is associated with later or current decline "suggests" where to look. It is not a proven driver.
- *Decision-support:* a ranked list helps a person decide what to review first. A human still checks each page.
- Any model result will be reported against the base rate and a transparent rule baseline, on held-out clients.

**What it will never say:**
- That a refresh *caused* a recovery. Without an experiment or a causal design this data cannot show that.
- That I found how Google ranks pages, or anything about AI citations or AI rankings. `ai_sessions_90d` only counts clicks from AI tools.
- That `trend_direction` is the truth about decline. It is a rule-made bucket from two 30-day windows, and it can reflect seasonality, noise or another page absorbing demand (consolidation).
- That results on the 30k starter slice carry over to the full warehouse. They would have to be earned again there.

**Known risks I am carrying forward:** label made from the current window (proxy), `avg_position = 0` means no data, missingness follows `content_type`, rate columns are x100 percentages, and no client names, URLs or queries appear in any output.

In [4]:
# Guard check: the label-source columns must never reach a feature list
LABEL_SOURCE = {"trend_direction", "trend_pct"}
ID_COLS = {"content_id", "client_id"}
candidate_features = [c for c in df.columns if c not in LABEL_SOURCE | ID_COLS]
assert not (set(candidate_features) & LABEL_SOURCE)
print(f"{len(candidate_features)} columns remain as candidate features (before a leakage review in the data-contract step)")
print("avg_position == 0 ('no data') rows:", int((df['avg_position'] == 0).sum()))

40 columns remain as candidate features (before a leakage review in the data-contract step)
avg_position == 0 ('no data') rows: 1205


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.